<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/Grupo6-D%C3%A9ficit%20em%20transa%C3%A7%C3%B5es%20correntes%20do%20PIB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 0) Instalação do pacote que consome a API do Banco Mundial
!pip install -q wbdata openpyxl

import wbdata
import pandas as pd
from google.colab import files

# ==============================================================================
# 0) OBTENÇÃO DOS DADOS VIA PACOTE DA API (wbdata)
# ==============================================================================
indicador_wb = {"BN.CAB.XOKA.GD.ZS": "valor"}
paises = ["ARG", "BRA", "CHL", "COL", "MEX"]

# Baixamos de 2019 a 2025 para ter os últimos 5 anos completos + 1 ano de defasagem
df_raw = wbdata.get_dataframe(
    indicators=indicador_wb,
    country=paises,
    date=("2019", "2025")
)

# ==============================================================================
# 1) TABELA BASE (FORMATO LONGO: periodo | indicador | valor)
# ==============================================================================
nomes_pt = {
    "Argentina": "Argentina",
    "Brazil": "Brasil",
    "Chile": "Chile",
    "Colombia": "Colômbia",
    "Mexico": "México"
}

tab_base = (
    df_raw.reset_index()
          .dropna(subset=["valor"])
          .assign(
              periodo=lambda x: x["date"].astype(int),
              indicador=lambda x: x["country"].map(nomes_pt)
          )[["periodo", "indicador", "valor"]]
          .sort_values(["indicador", "periodo"])
          .reset_index(drop=True)
)

# Garante apenas anos em que todos os 5 países já possuem dado consolidado
contagem_por_ano = tab_base.groupby("periodo")["valor"].count()
anos_completos = sorted(contagem_por_ano[contagem_por_ano == len(paises)].index)

# Mantém os últimos 6 anos completos (5 anos principais + 1 ano anterior para defasagem)
anos_trabalho = anos_completos[-6:]
tab_base_completa = tab_base[tab_base["periodo"].isin(anos_trabalho)].copy()

# Tabela Base final exibindo os últimos 5 anos
ultimos_5_anos = anos_completos[-5:]
TABELA_BASE = tab_base_completa[tab_base_completa["periodo"].isin(ultimos_5_anos)].reset_index(drop=True)

print("=== 1) TABELA BASE (Últimos 5 anos - Formato Longo) ===")
display(TABELA_BASE)

# ==============================================================================
# 2) FUNÇÕES PARA ACUMULADO NO ANO E ACUMULADO EM 12 MESES
# ==============================================================================
def acumulado_no_ano(df, tipo_variavel="fluxo", frequencia="anual"):
    """
    Calcula o acumulado no ano distinguindo tipo de variável ('fluxo', 'estoque', 'variacao')
    e frequência ('mensal', 'trimestral', 'anual').
    """
    res = df.copy().sort_values(["indicador", "periodo"])
    res["ano"] = res["periodo"].astype(str).str[:4].astype(int)

    if frequencia == "anual" or tipo_variavel == "estoque":
        # Série anual já representa o ano fechado; estoque reflete a posição do período
        res["valor"] = res["valor"]
    elif tipo_variavel == "fluxo":
        res["valor"] = res.groupby(["indicador", "ano"])["valor"].cumsum()
    elif tipo_variavel == "variacao":
        res["fator"] = 1 + (res["valor"] / 100.0)
        res["valor"] = (res.groupby(["indicador", "ano"])["fator"].cumprod() - 1) * 100.0
        res = res.drop(columns=["fator"])

    return res[["periodo", "indicador", "valor"]].reset_index(drop=True)


def acumulado_12_meses(df, tipo_variavel="fluxo", frequencia="anual"):
    """
    Calcula o acumulado em 12 meses distinguindo tipo de variável ('fluxo', 'estoque', 'variacao')
    e frequência ('mensal', 'trimestral', 'anual').
    """
    res = df.copy().sort_values(["indicador", "periodo"])
    janela = {"mensal": 12, "trimestral": 4, "anual": 1}[frequencia]

    if frequencia == "anual" or tipo_variavel == "estoque":
        # Série anual já representa 12 meses; estoque reflete a posição do período
        res["valor"] = res["valor"]
    elif tipo_variavel == "fluxo":
        res["valor"] = (
            res.groupby("indicador")["valor"]
               .rolling(window=janela, min_periods=janela)
               .sum()
               .reset_index(level=0, drop=True)
        )
    elif tipo_variavel == "variacao":
        res["fator"] = 1 + (res["valor"] / 100.0)
        res["valor"] = (
            res.groupby("indicador")["fator"]
               .rolling(window=janela, min_periods=janela)
               .apply(lambda x: x.prod() - 1, raw=True)
               .reset_index(level=0, drop=True)
        ) * 100.0
        res = res.drop(columns=["fator"])

    return res[["periodo", "indicador", "valor"]].reset_index(drop=True)

# ==============================================================================
# 3) TABELAS INTERMEDIÁRIAS (1, 2, 3 e 4)
# ==============================================================================
# Aplicamos sobre tab_base_completa (6 anos) para que o 1º dos 5 anos tenha ano-1
interm_1_full = acumulado_no_ano(tab_base_completa, tipo_variavel="fluxo", frequencia="anual")
interm_2_full = acumulado_12_meses(tab_base_completa, tipo_variavel="fluxo", frequencia="anual")

TABELA_INTERMEDIARIA_1 = interm_1_full[interm_1_full["periodo"].isin(ultimos_5_anos)].reset_index(drop=True)
TABELA_INTERMEDIARIA_2 = interm_2_full[interm_2_full["periodo"].isin(ultimos_5_anos)].reset_index(drop=True)


def gerar_tabela_variacao_anual(tab_interm):
    """
    Adiciona 'periodo_anterior' (ano - 1), faz o self-merge pela chave
    (indicador, periodo_anterior == periodo) e calcula valor / valor_y - 1.
    """
    atual = tab_interm.copy()
    atual["periodo_anterior"] = atual["periodo"] - 1

    passado = tab_interm[["periodo", "indicador", "valor"]].copy()

    merged = pd.merge(
        atual,
        passado,
        left_on=["indicador", "periodo_anterior"],
        right_on=["indicador", "periodo"],
        how="inner",
        suffixes=("", "_y")
    )

    # Fórmula solicitada no enunciado: valor / valor.y - 1
    merged["variacao_relativa"] = (merged["valor"] / merged["valor_y"]) - 1
    # Diferença em pontos percentuais (útil quando há troca de sinal entre déficit e superávit)
    merged["diferenca_pp"] = merged["valor"] - merged["valor_y"]

    return merged[["periodo", "indicador", "valor", "periodo_anterior", "valor_y", "variacao_relativa", "diferenca_pp"]]


TABELA_INTERMEDIARIA_3 = gerar_tabela_variacao_anual(interm_1_full)
TABELA_INTERMEDIARIA_3 = TABELA_INTERMEDIARIA_3[TABELA_INTERMEDIARIA_3["periodo"].isin(ultimos_5_anos)].reset_index(drop=True)

TABELA_INTERMEDIARIA_4 = gerar_tabela_variacao_anual(interm_2_full)
TABELA_INTERMEDIARIA_4 = TABELA_INTERMEDIARIA_4[TABELA_INTERMEDIARIA_4["periodo"].isin(ultimos_5_anos)].reset_index(drop=True)

print("\n=== 3) TABELA INTERMEDIÁRIA 3 (Variação vs. 12 meses antes) ===")
display(TABELA_INTERMEDIARIA_3.head(10))

# ==============================================================================
# 4) JUNÇÃO FINAL (LONG TO WIDE DAS 3 ÚLTIMAS OBS + INTERMEDIÁRIAS)
# ==============================================================================
ultimos_3_anos = ultimos_5_anos[-3:]
ultimo_periodo = ultimos_5_anos[-1]

# 4.1) Long to wide das 3 últimas observações da Tabela Base
wide_base_3 = TABELA_BASE[TABELA_BASE["periodo"].isin(ultimos_3_anos)].pivot(
    index="indicador",
    columns="periodo",
    values="valor"
)
wide_base_3.columns = [str(ano) for ano in wide_base_3.columns]

# 4.2) Coluna Acumulado no ano para o último período (Tabela Intermediária 1)
col_ac_ano = (
    TABELA_INTERMEDIARIA_1[TABELA_INTERMEDIARIA_1["periodo"] == ultimo_periodo]
    .set_index("indicador")[["valor"]]
    .rename(columns={"valor": f"Ac. no ano (até {ultimo_periodo})"})
)

# 4.3) Coluna Acumulado em 12 meses para o último período (Tabela Intermediária 2)
col_ac_12m = (
    TABELA_INTERMEDIARIA_2[TABELA_INTERMEDIARIA_2["periodo"] == ultimo_periodo]
    .set_index("indicador")[["valor"]]
    .rename(columns={"valor": f"Ac. em 12 meses (até {ultimo_periodo})"})
)

# 4.4) Colunas das Tabelas Intermediárias 3 e 4 para o último período
col_var_3 = (
    TABELA_INTERMEDIARIA_3[TABELA_INTERMEDIARIA_3["periodo"] == ultimo_periodo]
    .set_index("indicador")[["variacao_relativa"]]
    .rename(columns={"variacao_relativa": f"Var. Ac. Ano ({ultimo_periodo})"})
)

col_var_4 = (
    TABELA_INTERMEDIARIA_4[TABELA_INTERMEDIARIA_4["periodo"] == ultimo_periodo]
    .set_index("indicador")[["variacao_relativa"]]
    .rename(columns={"variacao_relativa": f"Var. 12m ({ultimo_periodo})"})
)

# Junção final
TABELA_FINAL = (
    wide_base_3
    .join(col_ac_ano)
    .join(col_ac_12m)
    .join(col_var_3)
    .join(col_var_4)
)

print("\n=== 4) TABELA FINAL CONSOLIDADA ===")
display(TABELA_FINAL)

# ==============================================================================
# EXPORTAÇÃO PARA EXCEL (TODAS AS TABELAS EM ABAS SEPARADAS)
# ==============================================================================
nome_arquivo = "exercicio_transacoes_correntes.xlsx"
with pd.ExcelWriter(nome_arquivo, engine="openpyxl") as writer:
    TABELA_FINAL.to_excel(writer, sheet_name="4_Tabela_Final")
    TABELA_BASE.to_excel(writer, sheet_name="1_Tabela_Base", index=False)
    TABELA_INTERMEDIARIA_1.to_excel(writer, sheet_name="3_Interm1_Ac_Ano", index=False)
    TABELA_INTERMEDIARIA_2.to_excel(writer, sheet_name="3_Interm2_Ac_12m", index=False)
    TABELA_INTERMEDIARIA_3.to_excel(writer, sheet_name="3_Interm3_Var_Ac_Ano", index=False)
    TABELA_INTERMEDIARIA_4.to_excel(writer, sheet_name="3_Interm4_Var_Ac_12m", index=False)

files.download(nome_arquivo)

=== 1) TABELA BASE (Últimos 5 anos - Formato Longo) ===


,periodo,indicador,valor
0,2021,Argentina,1.361544
1,2022,Argentina,-0.625255
2,2023,Argentina,-3.442033
3,2024,Argentina,0.922889
4,2025,Argentina,-1.140119
5,2021,Brasil,-2.358221
6,2022,Brasil,-2.149677
7,2023,Brasil,-1.231394
8,2024,Brasil,-2.988014
9,2025,Brasil,-2.926301



=== 3) TABELA INTERMEDIÁRIA 3 (Variação vs. 12 meses antes) ===


,periodo,indicador,valor,periodo_anterior,valor_y,variacao_relativa,diferenca_pp
0,2021,Argentina,1.361544,2020,0.696935,0.953616,0.664609
1,2022,Argentina,-0.625255,2021,1.361544,-1.459225,-1.986799
2,2023,Argentina,-3.442033,2022,-0.625255,4.505011,-2.816779
3,2024,Argentina,0.922889,2023,-3.442033,-1.268123,4.364922
4,2025,Argentina,-1.140119,2024,0.922889,-2.235380,-2.063008
5,2021,Brasil,-2.358221,2020,-1.641035,0.437032,-0.717186
6,2022,Brasil,-2.149677,2021,-2.358221,-0.088433,0.208544
7,2023,Brasil,-1.231394,2022,-2.149677,-0.427172,0.918283
8,2024,Brasil,-2.988014,2023,-1.231394,1.426529,-1.756620
9,2025,Brasil,-2.926301,2024,-2.988014,-0.020653,0.061713



=== 4) TABELA FINAL CONSOLIDADA ===


,2023,2024,2025,Ac. no ano (até 2025),Ac. em 12 meses (até 2025),Var. Ac. Ano (2025),Var. 12m (2025)
indicador,,,,,,,
Argentina,-3.442033,0.922889,-1.140119,-1.140119,-1.140119,-2.235380,-2.235380
Brasil,-1.231394,-2.988014,-2.926301,-2.926301,-2.926301,-0.020653,-0.020653
Chile,-3.104289,-1.168172,-1.216880,-1.216880,-1.216880,0.041696,0.041696
Colômbia,-2.268279,-1.688489,-2.409608,-2.409608,-2.409608,0.427079,0.427079
México,-0.570233,-0.917281,-0.510910,-0.510910,-0.510910,-0.443016,-0.443016


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>